# 10 - CLI Reference Manual

> **When to use**: When you want to operate directly in the terminal without writing Python code.
>
> **Core concept**: `sqlseed-cli` provides fill, preview, inspect, init and replay. The optional `sqlseed-ai` package registers AI commands; they require a separately configured model backend.

## Applicable Scenarios

- Quick fill test data → `sqlseed fill`
- Preview data without writing → `sqlseed preview`
- View column mapping strategies → `sqlseed inspect --show-mapping`
- AI generates config → `sqlseed ai-suggest`
- Replay snapshot → `sqlseed replay`

## What You Will Learn

- Common offline commands and config-driven filling
- Output formatting
- Error handling

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 01 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| **→ 10** | **CLI Reference Manual** | **CLI** | **06** |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [ ]:
from __future__ import annotations

# Run from examples/notebooks. Install from the repository root in one resolution:
# python -m pip install -e ".[dev,all]" -e "./plugins/sqlseed-cli" \
#   -e "./plugins/sqlseed-ai[dev,mcp]" -e "./plugins/mcp-server-sqlseed" -e "./plugins/sqlseed-web[dev]"
import os
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path

from click.testing import CliRunner
from sqlseed_cli import cli

import sqlseed
from sqlseed import connect

sys.path.insert(0, str(Path("..").resolve()))  # build_demo_db only
from build_demo_db import build

# Keep this object alive across cells. No existing database is opened or rebuilt.
_demo_directory = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-")
demo_root = Path(_demo_directory.name)
os.environ["SQLSEED_CACHE_DIR"] = str(demo_root / "cache")
db_path = build(demo_root / "demo.db")


def require(condition, message):
    """Stop the tutorial if an expected outcome did not occur."""
    if not condition:
        raise RuntimeError(message)


def check_generation(result, count):
    """Check errors and generated row count before showing success."""
    require(not result.errors and result.count == count, f"Generation failed: {result.errors}; count={result.count}")


def read_rows(database, sql):
    """Read actual persisted values using a fixed tutorial query."""
    # The queries below are fixed tutorial SQL, never external identifiers.
    with closing(sqlite3.connect(database)) as connection:
        return connection.execute(sql).fetchall()


with connect(str(db_path), provider="faker") as orch:
    for table, count in (("organizations", 5),):
        check_generation(orch.fill_table(table, count=count, seed=42, skip_ai=True), count)

print(f"sqlseed {sqlseed.__version__} | Temporary database: {db_path}")


os.environ["SQLSEED_LOG_LEVEL"] = "WARNING"
runner = CliRunner()


def run_cli(*args):
    """Run a CLI command and reject unexpected nonzero exits."""
    result = runner.invoke(cli, list(args))
    require(result.exit_code == 0, f"CLI failed ({result.exit_code}): {result.output}")
    return result.output

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| CLI Parsing | `plugins/sqlseed-cli/src/sqlseed_cli/main.py` | `cli()` |

## 1. See It in Action — Fill Data with One Command

No need to write Python code; a single CLI command generates data:

```bash
sqlseed fill app.db --table users --count 10000 --seed 42
```

Below we run common offline commands. Use `sqlseed --help` and each command's `--help` for the complete reference.

## 2. Quick Fill: One Command

Simplest usage: specify the database path, table name, and row count.

In [ ]:
print(
    run_cli(
        "fill", str(db_path), "-t", "members", "-n", "100", "--seed", "42", "--clear", "--provider", "faker", "--no-ai"
    )
)
require(len(read_rows(db_path, "SELECT member_id FROM members")) == 100, "CLI did not write 100 members")

## 3. Preview Data: No Database Write

The `preview` command generates data without writing, suitable for verifying mapping results.

In [ ]:
before = read_rows(db_path, "SELECT org_code, name FROM organizations ORDER BY org_code")
print(run_cli("preview", str(db_path), "-t", "organizations", "-n", "3", "--seed", "42", "--provider", "faker"))
require(
    read_rows(db_path, "SELECT org_code, name FROM organizations ORDER BY org_code") == before, "Preview wrote rows"
)

## 4. Inspect Schema and Column Mapping

`inspect` shows table structure; `--show-mapping` displays each column's mapping strategy.

In [ ]:
result = run_cli("inspect", str(db_path), "-t", "organizations", "--show-mapping")
print(result)

## 5. Generate Config Template: `init`

Auto-generates a YAML config based on the database schema; you can manually edit it and run with `fill -c`.

In [ ]:
from pathlib import Path

init_path = demo_root / "cli-demo-config.yaml"
result = run_cli("init", str(init_path), "--db", str(db_path))
print(result)

# Show generated config
print(init_path.read_text()[:500])
init_path.unlink(missing_ok=True)

## 6. Fill from Config File: `fill -c`

Batch-fill using a YAML/JSON config file, supporting advanced features like multi-table, custom generators, and cross-table associations.

In [ ]:
from sqlseed.config.loader import save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

# Create a config file
config = GeneratorConfig(
    db_path=str(db_path),
    provider="faker",
    tables=[
        TableConfig(name="tags", count=5, clear_before=True, seed=42),
    ],
)
config_path = demo_root / "cli-fill-config.yaml"
save_config(config, str(config_path))

# Fill from config
result = run_cli("fill", "-c", str(config_path), "--seed", "42")
print(result)
require(len(read_rows(db_path, "SELECT tag_id FROM tags")) == 5, "Config fill did not persist five tags")

config_path.unlink(missing_ok=True)

## 7. Error Handling and Exit Codes

CLI commands return a non-zero exit code to indicate an error.

In [ ]:
result = runner.invoke(cli, ["fill", str(db_path)])
require(result.exit_code != 0, "Missing table argument should fail")
print(f"Expected failure, exit code {result.exit_code}: {result.output.strip()[:200]}")

## Summary

| Command | Purpose | Writes to DB |
|------|------|:----------:|
| `fill` | Fill data | ✅ |
| `preview` | Preview without writing | ❌ |
| `inspect` | View schema | ❌ |
| `init` | Generate config template | ❌ |
| `fill -c` | Fill from config | ✅ |
| `replay` | Replay snapshot | ✅ |
| `ai-suggest` | AI generates config | ❌ |

**Next**: [11-utilities.ipynb](11-utilities.ipynb) — Utilities Reference

In [ ]:
require(len(read_rows(db_path, "SELECT member_id FROM members")) == 100, "Expected 100 members after the CLI workflow")
require(len(read_rows(db_path, "SELECT tag_id FROM tags")) == 5, "Expected five tags after config fill")
print("CLI exit codes, writes and non-writing preview verified.")